# 04 \u2014 Full Fuzzing Campaign (scaled run)

**Purpose.** Reproduce the **scaled** campaign behind `RESULTS.md`: ShadowPickle baseline (H1 denominator), then the guided (oracle-aware, adaptive evasion, 25 rounds x 20) and unguided (uniform random, 24 rounds x 20) campaigns seeded from a real text-generation checkpoint. All candidates land in the campaign DB.

This is the larger/full run. The quick pilot version (guided/unguided 5 rounds) lives in `04b_campaign_demo.ipynb` and is what the demo notebook references.

**Inputs / outputs**
- `real_benign_corpus/all/`
- panel images (picklescan/fickling/modelscan) + dynahug + oracle model dir

**Outputs**
- `data/regenbench_shadowpickle.db`
- `data/regenbench_campaign.db` (candidates, fitness, coverage)
- `docs/fuzzing-report-<run>.md`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_shadowpickle_baseline.py",
     "--candidates-per-family", "20", "--backend", "docker"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_fuzzing_campaign.py", "--mode", "guided",
     "--rounds", "25", "--candidates-per-round", "20", "--replicate", "1",
     "--db", "data/regenbench_campaign.db", "--seed-corpus-dir", "real_benign_corpus/all", "--seed-cluster", "text-generation",
     "--attack-families", "gadget,overwritten,pypi_injected,external,indirect_chain",
     "--evasion-mode", "adaptive", "--fitness-mode", "oracle_aware",
     "--backend", "docker", "--seed", "42"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_fuzzing_campaign.py", "--mode", "unguided",
     "--rounds", "24", "--candidates-per-round", "20", "--replicate", "1",
     "--db", "data/regenbench_campaign.db", "--seed-corpus-dir", "real_benign_corpus/all", "--seed-cluster", "text-generation",
     "--attack-families", "gadget,overwritten,pypi_injected,external,indirect_chain",
     "--evasion-mode", "random", "--fitness-mode", "current",
     "--backend", "docker", "--seed", "42"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
print("per-run generated / valid / bypasses (pt):")
print(sqlite("""SELECT c.run_id, COUNT(*) generated,
       SUM(CASE WHEN f.is_valid=1 AND COALESCE(c.format,'pt')='pt' THEN 1 ELSE 0 END) valid,
       SUM(CASE WHEN f.is_valid=1 AND COALESCE(c.format,'pt')='pt' AND c.panel_verdict='all_benign' THEN 1 ELSE 0 END) bypasses
FROM candidates c JOIN campaign_fitness f ON f.candidate_id=c.candidate_id
WHERE c.run_id IN ('guided-r1','unguided-r1')
GROUP BY c.run_id;"""))